In [1]:
# ==============================
# Gender Classification
# Dataset Loading
# ==============================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

# Handle notebook location
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

USERS_PATH = (
    PROJECT_ROOT
    / "travel_capstone_dataset"
    / "users.csv"
)

print("Dataset path:", USERS_PATH)
print("File exists:", USERS_PATH.exists())

users = pd.read_csv(USERS_PATH)

print("Shape:", users.shape)
print("Columns:", users.columns.tolist())

display(users.head())

Dataset path: c:\Users\VINAY\Desktop\Labmentix Projects\Travel_MLops_Major_Project\travel_capstone_dataset\users.csv
File exists: True
Shape: (1340, 5)
Columns: ['code', 'company', 'name', 'gender', 'age']


,code,company,name,gender,age
0,0,4You,Roy Braun,male,21
1,1,4You,Joseph Holsten,male,37
2,2,4You,Wilma Mcinnis,female,48
3,3,4You,Paula Daniel,female,23
4,4,4You,Patricia Carson,female,44


In [2]:
print("\nDataset information:")
users.info()

print("\nMissing values:")
print(users.isnull().sum())

print("\nDuplicate rows:", users.duplicated().sum())

print("\nGender distribution:")
print(users["gender"].value_counts(dropna=False))


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   code     1340 non-null   int64 
 1   company  1340 non-null   object
 2   name     1340 non-null   object
 3   gender   1340 non-null   object
 4   age      1340 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 52.5+ KB

Missing values:
code       0
company    0
name       0
gender     0
age        0
dtype: int64

Duplicate rows: 0

Gender distribution:
gender
male      452
female    448
none      440
Name: count, dtype: int64


In [3]:
# ==============================
# Gender Classification
# Train/Test Preparation
# ==============================

from sklearn.model_selection import train_test_split

TARGET = "gender"

X = users.drop(columns=[TARGET])
y = users[TARGET]

print("Features:", X.columns.tolist())
print("Target:", TARGET)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training distribution:")
print(y_train.value_counts())

Features: ['code', 'company', 'name', 'age']
Target: gender
Training samples: 1072
Testing samples: 268
Training distribution:
gender
male      362
female    358
none      352
Name: count, dtype: int64


In [4]:
# ==============================
# Preprocessing Pipeline
# ==============================

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
import numpy as np

def identity_1d(x):
    return np.asarray(x).ravel()

name_pipeline = Pipeline([
    (
        "to_1d",
        FunctionTransformer(identity_1d, validate=False)
    ),
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5),
            min_df=2
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "company",
            OneHotEncoder(handle_unknown="ignore"),
            ["company"]
        ),
        (
            "name",
            name_pipeline,
            ["name"]
        ),
        (
            "age",
            StandardScaler(),
            ["age"]
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [5]:
# ==============================
# Logistic Regression Baseline
# ==============================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(
    y_test,
    y_pred_lr,
    average="weighted",
    zero_division=0
)
recall_lr = recall_score(
    y_test,
    y_pred_lr,
    average="weighted",
    zero_division=0
)
f1_lr = f1_score(
    y_test,
    y_pred_lr,
    average="weighted",
    zero_division=0
)

print("Logistic Regression Performance")
print("-" * 40)
print(f"Accuracy : {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall   : {recall_lr:.4f}")
print(f"F1 Score : {f1_lr:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Performance
----------------------------------------
Accuracy : 0.5560
Precision: 0.5435
Recall   : 0.5560
F1 Score : 0.5463

Classification Report:
              precision    recall  f1-score   support

      female       0.61      0.60      0.61        90
        male       0.61      0.73      0.66        90
        none       0.41      0.33      0.36        88

    accuracy                           0.56       268
   macro avg       0.54      0.55      0.54       268
weighted avg       0.54      0.56      0.55       268



In [6]:
# ==============================
# Random Forest Classifier
# ==============================

from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )
    )
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)
recall_rf = recall_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)
f1_rf = f1_score(
    y_test,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

print("Random Forest Performance")
print("-" * 40)
print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Performance
----------------------------------------
Accuracy : 0.5299
Precision: 0.5118
Recall   : 0.5299
F1 Score : 0.5187

Classification Report:
              precision    recall  f1-score   support

      female       0.60      0.62      0.61        90
        male       0.61      0.70      0.65        90
        none       0.32      0.26      0.29        88

    accuracy                           0.53       268
   macro avg       0.51      0.53      0.52       268
weighted avg       0.51      0.53      0.52       268



In [7]:
# ==============================
# XGBoost Classifier
# ==============================

from xgboost import XGBClassifier

# Encode target labels for XGBoost
label_map = {
    "female": 0,
    "male": 1,
    "none": 2
}

y_train_xgb = y_train.map(label_map)
y_test_xgb = y_test.map(label_map)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softmax",
            num_class=3,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1
        )
    )
])

xgb_pipeline.fit(X_train, y_train_xgb)

y_pred_xgb_encoded = xgb_pipeline.predict(X_test)

reverse_map = {
    0: "female",
    1: "male",
    2: "none"
}

y_pred_xgb = pd.Series(y_pred_xgb_encoded).map(reverse_map)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(
    y_test, y_pred_xgb,
    average="weighted",
    zero_division=0
)
recall_xgb = recall_score(
    y_test, y_pred_xgb,
    average="weighted",
    zero_division=0
)
f1_xgb = f1_score(
    y_test, y_pred_xgb,
    average="weighted",
    zero_division=0
)

print("XGBoost Performance")
print("-" * 40)
print(f"Accuracy : {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall   : {recall_xgb:.4f}")
print(f"F1 Score : {f1_xgb:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Performance
----------------------------------------
Accuracy : 0.5149
Precision: 0.4899
Recall   : 0.5149
F1 Score : 0.4969

Classification Report:
              precision    recall  f1-score   support

      female       0.60      0.61      0.60        90
        male       0.56      0.71      0.63        90
        none       0.31      0.22      0.25        88

    accuracy                           0.51       268
   macro avg       0.49      0.51      0.50       268
weighted avg       0.49      0.51      0.50       268



In [9]:
import sentence_transformers

print("sentence-transformers:", sentence_transformers.__version__)

c:\Users\VINAY\Desktop\Labmentix Projects\Travel_MLops_Major_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers: 6.0.0


In [10]:
# ==============================
# Name Embedding Model
# ==============================

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1246.70it/s]


Embedding model loaded successfully.


In [11]:
# ==============================
# Generate Name Embeddings
# ==============================

train_names = X_train["name"].astype(str).tolist()
test_names = X_test["name"].astype(str).tolist()

X_train_name_emb = embedding_model.encode(
    train_names,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_name_emb = embedding_model.encode(
    test_names,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Train embedding shape:", X_train_name_emb.shape)
print("Test embedding shape :", X_test_name_emb.shape)

Batches: 100%|██████████| 9/9 [00:01<00:00,  5.22it/s]

Train embedding shape: (1072, 384)
Test embedding shape : (268, 384)


## Step 1 — PCA

In [12]:
# ==============================
# PCA on Name Embeddings
# ==============================

from sklearn.decomposition import PCA

pca = PCA(
    n_components=50,
    random_state=42
)

X_train_name_pca = pca.fit_transform(X_train_name_emb)
X_test_name_pca = pca.transform(X_test_name_emb)

print("PCA train shape:", X_train_name_pca.shape)
print("PCA test shape :", X_test_name_pca.shape)

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_.sum()
)

PCA train shape: (1072, 50)
PCA test shape : (268, 50)
Explained variance ratio: 0.6756655


## Step 2 — Encode company and scale age

In [13]:
# ==============================
# Company + Age Features
# ==============================

from sklearn.preprocessing import OneHotEncoder, StandardScaler

company_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_company = company_encoder.fit_transform(
    X_train[["company"]]
)

X_test_company = company_encoder.transform(
    X_test[["company"]]
)

age_scaler = StandardScaler()

X_train_age = age_scaler.fit_transform(
    X_train[["age"]]
)

X_test_age = age_scaler.transform(
    X_test[["age"]]
)

print("Company train shape:", X_train_company.shape)
print("Age train shape:", X_train_age.shape)

Company train shape: (1072, 5)
Age train shape: (1072, 1)


## Step 3 — Combine everything

In [14]:
# ==============================
# Final Feature Matrix
# ==============================

import numpy as np

X_train_final = np.hstack([
    X_train_name_pca,
    X_train_company,
    X_train_age
])

X_test_final = np.hstack([
    X_test_name_pca,
    X_test_company,
    X_test_age
])

print("Final train shape:", X_train_final.shape)
print("Final test shape :", X_test_final.shape)

Final train shape: (1072, 56)
Final test shape : (268, 56)


## Step 4 — Retrain Logistic Regression

In [15]:
# ==============================
# Logistic Regression
# Transformer + PCA
# ==============================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

lr_name_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

lr_name_model.fit(
    X_train_final,
    y_train
)

y_pred_lr_name = lr_name_model.predict(
    X_test_final
)

accuracy_lr_name = accuracy_score(
    y_test,
    y_pred_lr_name
)

precision_lr_name = precision_score(
    y_test,
    y_pred_lr_name,
    average="weighted",
    zero_division=0
)

recall_lr_name = recall_score(
    y_test,
    y_pred_lr_name,
    average="weighted",
    zero_division=0
)

f1_lr_name = f1_score(
    y_test,
    y_pred_lr_name,
    average="weighted",
    zero_division=0
)

print("Logistic Regression — Transformer + PCA")
print("-" * 50)
print(f"Accuracy : {accuracy_lr_name:.4f}")
print(f"Precision: {precision_lr_name:.4f}")
print(f"Recall   : {recall_lr_name:.4f}")
print(f"F1 Score : {f1_lr_name:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_lr_name,
        zero_division=0
    )
)

Logistic Regression — Transformer + PCA
--------------------------------------------------
Accuracy : 0.6455
Precision: 0.5916
Recall   : 0.6455
F1 Score : 0.5870

Classification Report:
              precision    recall  f1-score   support

      female       0.69      0.86      0.76        90
        male       0.66      0.92      0.77        90
        none       0.42      0.15      0.22        88

    accuracy                           0.65       268
   macro avg       0.59      0.64      0.58       268
weighted avg       0.59      0.65      0.59       268



In [16]:
# ==============================
# PCA - 100 Components
# ==============================

pca_100 = PCA(
    n_components=100,
    random_state=42
)

X_train_name_pca_100 = pca_100.fit_transform(
    X_train_name_emb
)

X_test_name_pca_100 = pca_100.transform(
    X_test_name_emb
)

print(
    "Explained variance:",
    pca_100.explained_variance_ratio_.sum()
)

print(
    "Train shape:",
    X_train_name_pca_100.shape
)

print(
    "Test shape:",
    X_test_name_pca_100.shape
)

Explained variance: 0.8610475
Train shape: (1072, 100)
Test shape: (268, 100)


In [17]:
X_train_final_100 = np.hstack([
    X_train_name_pca_100,
    X_train_company,
    X_train_age
])

X_test_final_100 = np.hstack([
    X_test_name_pca_100,
    X_test_company,
    X_test_age
])

print("Final train shape:", X_train_final_100.shape)
print("Final test shape :", X_test_final_100.shape)

Final train shape: (1072, 106)
Final test shape : (268, 106)


In [18]:
lr_name_model_100 = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

lr_name_model_100.fit(
    X_train_final_100,
    y_train
)

y_pred_lr_100 = lr_name_model_100.predict(
    X_test_final_100
)

accuracy_lr_100 = accuracy_score(y_test, y_pred_lr_100)
precision_lr_100 = precision_score(
    y_test,
    y_pred_lr_100,
    average="weighted",
    zero_division=0
)
recall_lr_100 = recall_score(
    y_test,
    y_pred_lr_100,
    average="weighted",
    zero_division=0
)
f1_lr_100 = f1_score(
    y_test,
    y_pred_lr_100,
    average="weighted",
    zero_division=0
)

print("Logistic Regression — Transformer + PCA (100)")
print("-" * 55)
print(f"Accuracy : {accuracy_lr_100:.4f}")
print(f"Precision: {precision_lr_100:.4f}")
print(f"Recall   : {recall_lr_100:.4f}")
print(f"F1 Score : {f1_lr_100:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_lr_100,
        zero_division=0
    )
)

Logistic Regression — Transformer + PCA (100)
-------------------------------------------------------
Accuracy : 0.6493
Precision: 0.6020
Recall   : 0.6493
F1 Score : 0.5968

Classification Report:
              precision    recall  f1-score   support

      female       0.70      0.87      0.78        90
        male       0.66      0.90      0.76        90
        none       0.44      0.17      0.25        88

    accuracy                           0.65       268
   macro avg       0.60      0.65      0.59       268
weighted avg       0.60      0.65      0.60       268



In [19]:
# ==============================
# Save Final Gender Classifier
# ==============================

from pathlib import Path
import joblib

# Project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

GENDER_MODEL_PATH = MODEL_DIR / "gender_classifier.joblib"

gender_model_bundle = {
    "embedding_model_name": "all-MiniLM-L6-v2",
    "embedding_model": embedding_model,
    "pca": pca_100,
    "company_encoder": company_encoder,
    "age_scaler": age_scaler,
    "classifier": lr_name_model_100,
    "classes": list(lr_name_model_100.classes_),
    "metrics": {
        "accuracy": accuracy_lr_100,
        "precision": precision_lr_100,
        "recall": recall_lr_100,
        "f1": f1_lr_100,
    },
}

joblib.dump(
    gender_model_bundle,
    GENDER_MODEL_PATH
)

print("Gender model saved successfully.")
print("Model path:", GENDER_MODEL_PATH)
print("File exists:", GENDER_MODEL_PATH.exists())

Gender model saved successfully.
Model path: c:\Users\VINAY\Desktop\Labmentix Projects\Travel_MLops_Major_Project\models\gender_classifier.joblib
File exists: True


In [20]:
# ==============================
# Gender Prediction Function
# ==============================

def predict_gender(name, company, age):
    name_embedding = embedding_model.encode(
        [str(name)],
        convert_to_numpy=True
    )

    name_pca = pca_100.transform(name_embedding)

    company_features = company_encoder.transform(
        pd.DataFrame({"company": [company]})
    )

    age_features = age_scaler.transform(
        pd.DataFrame({"age": [age]})
    )

    final_features = np.hstack([
        name_pca,
        company_features,
        age_features
    ])

    prediction = lr_name_model_100.predict(final_features)

    return prediction[0]

In [21]:
# ==============================
# Test Predictions
# ==============================

test_cases = [
    ("John Smith", "4You", 30),
    ("Priya Sharma", "4You", 28),
    ("Wilma Mcinnis", "4You", 48),
]

for name, company, age in test_cases:
    result = predict_gender(
        name=name,
        company=company,
        age=age
    )

    print(
        f"Name: {name:20s} | "
        f"Company: {company:10s} | "
        f"Age: {age:2d} | "
        f"Prediction: {result}"
    )

Name: John Smith           | Company: 4You       | Age: 30 | Prediction: male
Name: Priya Sharma         | Company: 4You       | Age: 28 | Prediction: female
Name: Wilma Mcinnis        | Company: 4You       | Age: 48 | Prediction: female


In [22]:
# ==============================
# Reload Saved Model
# ==============================

loaded_gender_bundle = joblib.load(
    GENDER_MODEL_PATH
)

print("Saved gender model loaded successfully.")
print("Classes:", loaded_gender_bundle["classes"])
print("Accuracy:", loaded_gender_bundle["metrics"]["accuracy"])

Saved gender model loaded successfully.
Classes: ['female', 'male', 'none']
Accuracy: 0.6492537313432836
